# Data Pipeline

### This notebook will clean and format the data for analysis
March 5th, 2025

Revised May 29th, 2025

Note: 
    Script will take ample time to run if using entire dataset

Maxime Bouthillier

### Importing Libraries and Specialty Functions 

In [1]:
import pandas as pd
import numpy as np
import os 
import glob 
from datetime import datetime
import warnings
from joblib import dump
import Pipeline_Functions as func
import matplotlib.pyplot as plt

# Note: Currently using DEMO dataset
directory = '/work/mbouthil/Research_Project/MIMIC-III_data/MIMIC-III_demo'
os.chdir(directory)
cwd = os.getcwd()
print("Current working directory:", cwd)

Current working directory: /mnt/hpc/work/mbouthil/Research_Project/MIMIC-III_data/MIMIC-III_demo


# Data Cleaning

### Admissions Table

In [3]:
# Reading the csv file
adm_df = pd.read_csv("ADMISSIONS.csv")
adm_df.columns = adm_df.columns.str.lower()


# Cleaning the time based features
adm_df["edregtime"] = adm_df["edregtime"].fillna("1677-09-22 00:00:00")                                             
adm_df["edouttime"] = adm_df["edouttime"].fillna("1677-09-22 00:00:00")

adm_df = func.as_datetime(adm_df, column='admittime')
adm_df = func.as_datetime(adm_df, column='dischtime')
adm_df = func.as_datetime(adm_df, column='edregtime')
adm_df = func.as_datetime(adm_df, column='edouttime')


# Removing all admission instances where a patient died
adm_df = adm_df.drop(adm_df[adm_df['hospital_expire_flag'] == 1].index)


# Setting Marital Status to binary variables
adm_df['marital_status'] = adm_df['marital_status'].apply(lambda x: "MARRIED" if x == "MARRIED" else "SINGLE")

In [4]:
# Creating Readmission Feature and subsetting the dataset
adm_df = func.readmission(adm_df, 30)

# Admission Duration time feature
adm_df['admit_duration'] = adm_df['dischtime'] - adm_df['admittime']

# ED Duration time feature
adm_df['ed_duration'] = adm_df['edouttime'] - adm_df['edregtime']


# Removing unnecessary variables
col_drops = ["row_id", "language", "religion", "hospital_expire_flag", "hadm_id", 
             "has_chartevents_data", "edregtime", "edouttime", "deathtime", "diagnosis"]

for i in col_drops:
    adm_df = adm_df.drop(i, axis=1)

# Checing NaN instances
func.check_nan(adm_df)

In [5]:
for col, col_values in adm_df.items():
    if adm_df[col].dtype == ('O'):
        adm_df[col] = adm_df[col].astype('category').cat.codes
       
    if adm_df[col].dtype == ('<m8[ns]'):
        adm_df[col] = adm_df[col].dt.total_seconds()
        adm_df[col] = adm_df[col]/(60**2*24)

### Patients Table

In [43]:
# Reading the csv file
pat_df = pd.read_csv("PATIENTS.csv")
pat_df.columns = pat_df.columns.str.lower()
pat_df = func.as_datetime(pat_df, column='dob')


# Selecting only the necessary columns
pat_df = pat_df[['subject_id', 'gender', 'dob']]


# Double checking that there are no NaN values
func.check_nan(pat_df)

### ICU LOS

In [44]:
# Reading the csv file
icu_df = pd.read_csv("ICUSTAYS.csv")
icu_df.columns = icu_df.columns.str.lower()
icu_df = icu_df.dropna()


# Cleaning the time based features
icu_df  = func.as_datetime(icu_df , column='intime')
icu_df = func.as_datetime(icu_df , column='outtime')


# Subsetting the dataframe by only the releveat subject_ID entries:
icu_df = func.subject_subset(icu_df , adm_df, column='intime')

In [45]:
# Keeping only the necessary columns
icu_df  = icu_df[['subject_id', 'los']]

# Combining discontinuous ICU stays
icu_df = icu_df.groupby('subject_id', as_index=False).agg({'los': 'sum'})

# Double checking that there are no NaN values
func.check_nan(icu_df)

### Output Events Table

In [46]:
# Cleaning + Formating
oe_df = pd.read_csv("OUTPUTEVENTS.csv")
oe_df.columns = oe_df.columns.str.lower()

oe_df = func.as_datetime(oe_df, column='charttime')
oe_df = func.subject_subset(oe_df, adm_df, column='charttime')

oe_df = oe_df[['subject_id', 'itemid', 'value']]
oe_df = oe_df.dropna()
func.check_nan(oe_df)

### Chart Events Table

In [47]:
# Cleaning + Formating
ce_df = pd.read_csv("CHARTEVENTS.csv")
ce_df = func.as_datetime(ce_df, column='charttime')

ce_df = func.subject_subset(ce_df, adm_df, column='charttime')
ce_df = ce_df.dropna()

ce_df = ce_df[['subject_id', 'itemid', 'value']]
func.check_nan(ce_df)

/tmp/ipykernel_4172686/1268021451.py:2: DtypeWarning: Columns (8,10,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  ce_df = pd.read_csv("CHARTEVENTS.csv")


### Events

In [48]:
# Combining Ouput Events and Chart Events
events_df = pd.concat([oe_df, ce_df])
events_df['itemid'] = events_df['itemid'].astype('category').cat.codes

events_df['value'] = pd.to_numeric(events_df['value'], errors='coerce').fillna(1)
events_df ['value'] = events_df['value'].replace(0, 1)


/tmp/ipykernel_4172686/4031083830.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events_df = pd.concat([oe_df, ce_df])


In [49]:
subjects = adm_df['subject_id']
event_col = []

for subject in subjects:

    # events
    event = events_df.loc[events_df['subject_id'] == subject]
    event = event.pivot_table(
        columns='itemid',         # one column per drug-code
        values='value',           # the cell value is the dose
        aggfunc='mean'            # in case there are duplicates taking summation
    )

    event = event.reindex(
    columns=np.arange(1,len(events_df['itemid'].unique()+1)),
    fill_value=0
    ).T

    event_col.append(event)

adm_df['event'] = event_col

### Inputs

In [50]:
# Input Events CV
cv_df = pd.read_csv("INPUTEVENTS_CV.csv")
cv_df = func.as_datetime(cv_df, column='charttime')
cv_df = func.subject_subset(cv_df, adm_df, column='charttime')
cv_df = cv_df[['subject_id', 'itemid', 'amount']] # , 'amountuom']]

cv_df = cv_df.dropna()
func.check_nan(cv_df)


# Input Events MV
mv_df = pd.read_csv("INPUTEVENTS_MV.csv")
mv_df = func.as_datetime(mv_df, column='starttime')
ie_df = func.subject_subset(mv_df, adm_df, column='starttime')
mv_df = mv_df[['subject_id', 'itemid', 'amount']] #, 'amountuom']]

func.check_nan(mv_df)

/tmp/ipykernel_4172686/1100881843.py:2: DtypeWarning: Columns (17,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  cv_df = pd.read_csv("INPUTEVENTS_CV.csv")


In [51]:
# Combining Input Events CV and Input Events MV
inputs_df = pd.concat([cv_df, mv_df])
inputs_df['itemid'] = inputs_df['itemid'].astype('category').cat.codes
inputs_df ['amount'] = inputs_df['amount'].replace(0, 1)

subjects = adm_df['subject_id']
input_col = []

for subject in subjects:

    # Procedure
    input = inputs_df.loc[inputs_df['subject_id'] == subject]
    input = input.pivot_table(
        columns='itemid',         # one column per drug-code
        values='amount',          # the cell value is the dose          
        aggfunc='mean'            # in case there are duplicates taking average
    )

    input = input.reindex(
    columns=np.arange(1,len(inputs_df['itemid'].unique()+1)),
    fill_value=0
    ).T

    input_col.append(input)

adm_df['input'] = input_col

### Lab Events

$ \textcolor{OrangeRed}{\text{Complete cleaning of data}}$ 

In [52]:
# Cleaning + Formating
labs_df = pd.read_csv("LABEVENTS.csv")
labs_df = func.as_datetime(labs_df, column='charttime')
labs_df = func.subject_subset(labs_df, adm_df, column='charttime')

labs_df = labs_df[['subject_id', 'itemid', 'value']]
labs_df = labs_df.dropna()

In [53]:
strings = labs_df['value'][labs_df['value'].apply(lambda x: isinstance(x, str))]
print(strings)

54193        15
54194        21
54195       8.0
54196       122
54197       0.7
          ...  
59648    Yellow
59649      RARE
59650       NEG
59651        80
59652      NONE
Name: value, Length: 30672, dtype: object


In [54]:
strings.unique()

array(['15', '21', '8.0', ..., '301', '0.78', 'LESS THAN 0.01'],
      dtype=object)

In [55]:
# lab_col = []

# for subject in subjects:

#     # Labs
#     lab = labs_df.loc[labs_df['subject_id'] == subject]
#     lab = lab.pivot_table(
#         columns='itemid',         # one column per drug-code
#         values='value',           # the cell value is the dose          
#         aggfunc='mean'            # in case there are duplicates taking summation
#     )

#     lab = lab.reindex(
#     columns=np.arange(1,len(labs_df['itemid'].unique()+1)),
#     fill_value=0
#     ).T

#     lab_col.append(lab)

# adm_df['lab'] = lab_col

### Prescriptions

$ \textcolor{OrangeRed}{\text{Complete cleaning of data}}$ 

In [56]:
# Reading the csv file
pres_df = pd.read_csv("PRESCRIPTIONS.csv")


# Cleaning the time based features
pres_df["enddate"] = pres_df["enddate"].fillna("1677-09-22 00:00:00")    
pres_df = func.as_datetime(pres_df, column='enddate')


# Subsetting the dataframe by only the releveat subject_ID entries:
pres_df = func.subject_subset(pres_df, adm_df, column='enddate')


#Selecting only the relevant columns
pres_df = pres_df[['subject_id', 'drug', 'dose_val_rx','dose_unit_rx', 'form_val_disp']]


# Double checking that there are no NaN values
func.check_nan(pres_df)

# standardixing ml
pres_df['dose_unit_rx'] = pres_df['dose_unit_rx'].replace(
    to_replace=r'(?i).*?\bml\b.*',  # any string containing "ml"
    value='ml',
    regex=True
)

# standardizing mcg/hr abd mcg/h to mcg/h
pres_df['dose_unit_rx'] = pres_df['dose_unit_rx'].replace(
    to_replace=r'(?i).*?\bmcg/?h(?:r)?\b.*',
    value='mcg/h',
    regex=True
)

pres_df['dose_unit_rx'] = pres_df['dose_unit_rx'].replace(
    to_replace=r'(?i).*?\bmcg/?h(?:r)?\b.*',
    value='mcg/h',
    regex=True
)

pres_df['dose_unit_rx'] = pres_df['dose_unit_rx'].replace(
    to_replace   = r'(?i).*?\bgm\b.*',   # any string containing “dose”, case‐insensitive
    value        = 'g',
    regex        = True
)

# These convertes all codes to categorical
# pres_df['drug'] = pres_df['drug'].astype('category').cat.codes

$ \textcolor{OrangeRed}{\text{The following section is incomplete}} $

In [57]:
# Precise formating of cell values

# standadizing ml and mL
pres_df['dose_unit_rx'] = pres_df['dose_unit_rx'].replace(
    to_replace=r'(?i).*?\bml\b.*',  # any string containing "ml"
    value='ml',
    regex=True
)

# standardizing mcg/hr and mcg/h
pres_df['dose_unit_rx'] = pres_df['dose_unit_rx'].replace(
    to_replace=r'(?i).*?\bmcg/?h(?:r)?\b.*',
    value='mcg/h',
    regex=True
)

# standardizing g and gm
pres_df['dose_unit_rx'] = pres_df['dose_unit_rx'].replace(
    to_replace   = r'(?i).*?\bgm\b.*', 
    value        = 'g',
    regex        = True
)

In [58]:
pres_df['dose_unit_rx'].unique()

array(['NEB', 'mg', 'Enema', 'g', 'UNIT', 'mcg', 'TAB', 'mcg/h', 'mEq',
       'ml', 'CAP', 'BAG', 'VIAL', 'mmol', 'PTCH', 'PKT', 'Appl', 'DROP',
       'LOZ', 'PUFF', 'dose', 'INH', 'SPRY', 'gtt', 'SYR', 'in',
       'million units', 'TROC', 'AMP', 'L'], dtype=object)

In [59]:
drugs = pres_df['drug'].unique()

diff_units = []

for drug in drugs:
    
    if len(pres_df.loc[pres_df['drug'] == drug, 'dose_unit_rx'].unique()) > 1:
        diff_units.append(drug)

In [60]:
diff_units

['Multivitamins',
 'Warfarin',
 '0.9% Sodium Chloride',
 'Fentanyl Citrate',
 'Senna',
 'Acetylcysteine 20%',
 'Oxycodone-Acetaminophen',
 'Sodium Bicarbonate',
 'Syringe',
 'Norepinephrine',
 'Octreotide Acetate',
 'Xopenex Neb',
 'Fludrocortisone Acetate',
 'Lidocaine']

In [61]:
pres_df.loc[pres_df['drug'] == diff_units[4], 'dose_unit_rx'].unique()

array(['TAB', 'ml'], dtype=object)

In [62]:
pres_df.loc[pres_df['drug'] == diff_units[4]]

,subject_id,drug,dose_val_rx,dose_unit_rx,form_val_disp
5663,10127,Senna,1,TAB,1
6114,10026,Senna,1,TAB,1
6115,10026,Senna,1,TAB,1
1995,40286,Senna,1,TAB,1
5849,10083,Senna,1,TAB,1
1172,10124,Senna,1,TAB,1
1186,10124,Senna,2,TAB,2
1671,40601,Senna,1,TAB,1
610,44222,Senna,1,TAB,1
5285,10094,Senna,1,TAB,1


In [63]:
pres_df.loc[pres_df['subject_id'] == 42231]

,subject_id,drug,dose_val_rx,dose_unit_rx,form_val_disp
8702,42231,Pneumococcal Vac Polyvalent,0.5,ml,1
8703,42231,Sodium Chloride 0.9% Flush,3,ml,0.6
8704,42231,Acetaminophen,650,mg,2
8705,42231,DiphenhydrAMINE,12.5,mg,0.5
8706,42231,Hydrocodone-Acetaminophen,1-2,TAB,1-2
8707,42231,Aluminum-Magnesium Hydrox.-Simethicone,15-30,ml,0.5-1
8708,42231,Docusate Sodium,100,mg,1
8709,42231,Ondansetron,4,mg,1
8710,42231,HydrALAzine,10,mg,0.5
8711,42231,Metoprolol Tartrate,5,mg,1


In [64]:
pres_df.loc[pres_df['drug'] == '0.9% Sodium Chloride']

,subject_id,drug,dose_val_rx,dose_unit_rx,form_val_disp
6762,41976,0.9% Sodium Chloride,1000,ml,1000
6768,41976,0.9% Sodium Chloride,1000,ml,1000
6772,41976,0.9% Sodium Chloride,1,BAG,1
6773,41976,0.9% Sodium Chloride,1,BAG,1
6774,41976,0.9% Sodium Chloride,2,BAG,2
...,...,...,...,...,...
645,43879,0.9% Sodium Chloride,2,BAG,2
647,43879,0.9% Sodium Chloride,500,ml,500
648,43879,0.9% Sodium Chloride,1000,ml,1
8744,42231,0.9% Sodium Chloride,1000,ml,1


In [65]:
# subject = 41976

# test = pres_df.loc[pres_df['subject_id'] == subject]
# test = test.pivot_table(
#         columns='drug',         # one column per drug-code
#         values='dose_val_rx',   # the cell value is the dose
#         fill_value=0,           # missing → 0
#         aggfunc='sum'           # in case there are duplicates taking summation
#     )
# test = test.reindex(
#     columns=np.arange(1,len(pres_df['drug'].unique())),
#     fill_value=0
#     ).T
# display(test)
# test.iloc[19,]

### Procedures

In [6]:
# Reading the csv file
pro_df = pd.read_csv("PROCEDUREEVENTS_MV.csv")

# Cleaning the time based features
pro_df = func.as_datetime(pro_df, column='starttime')
pro_df = func.as_datetime(pro_df, column='endtime')
pro_df["duration"] = (pro_df['endtime'] - pro_df['starttime']).dt.total_seconds()/60

# Subsetting the dataframe by subject_IDs:
pro_df = func.subject_subset(pro_df, adm_df, column='starttime')
pro_df = pro_df[['subject_id','itemid','duration']]
pro_df = pro_df.dropna()

# These convertes all codes to categorical
pro_df['itemid'] = pro_df['itemid'].astype('category').cat.codes

In [7]:
# Subject Level Formating

subjects = adm_df['subject_id']
proc_col = []

for subject in subjects:

    # Procedure
    procedure = pro_df.loc[pro_df['subject_id'] == subject]
    procedure = procedure.pivot_table(
        columns='itemid',         # one column per drug-code
        values='duration',        # the cell value is the dose
        aggfunc='mean'            # in case there are duplicates taking summation
    )

    procedure = procedure.reindex(
    columns=np.arange(1,len(pro_df['itemid'].unique()+1)),
    fill_value=0
    ).T

    proc_col.append(procedure)

adm_df['procedure'] = proc_col

In [8]:
pro_df.head()

,subject_id,itemid,duration
446,43798,36,6571.0
447,43798,11,5086.0
448,43798,13,1951.0
449,43798,19,1.0
450,43798,33,1.0


# Combining Tables

In [68]:
# Final Assembly of data
master_df = pd.merge(adm_df, pat_df, how='inner', on='subject_id')          # Left join of Patients table on Admission table
master_df = pd.merge(master_df, icu_df, how='inner', on='subject_id')       # Left join of ICU LOS table on df
master_df = master_df.fillna(0)

# Creatinon of the Age variable and removal of the dob column
master_df['age'] = master_df['admittime'].dt.year - master_df['dob'].dt.year
master_df = master_df.drop('dob', axis=1)

In [69]:
master_df.loc[master_df['subject_id'] == 10127, 'input'].iloc[0]

,amount
itemid,
1,0.000000
2,0.000000
3,115.864026
4,86.066667
5,17.301217
...,...
186,0.000000
187,0.000000
188,0.000000


In [70]:
# Old codes.

# # Gathering subjects
# subjects = list(master_df['subject_id'])

# # Combining all data into one table
# for i in range(len(tables)):

#     new_column = []
#     table = tables[i]

#     for j in subjects:
#         data = table.loc[table['subject_id'] ==  j]
#         data = data.drop('subject_id',axis=1)
#         new_column.append(data)

#     master_df[col_name[i]] = new_column

# master_df = master_df.fillna(0)

# # Dropping the unnecessary columns once combined
# master_df = master_df.drop(['admittime', 'dischtime'], axis=1)

In [71]:
master_df.head(2)

,subject_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,marital_status,ethnicity,read_flag,admit_duration,ed_duration,event,input,procedure,gender,los,age
0,42231,2102-08-29 07:15:00,2102-09-06 16:20:00,0,2,2,1,0,7,0.0,8.378472,0.000000,value itemid 1 0.00...,amount itemid 1 0.0 2...,duration itemid 1 ...,F,1.1358,86
1,43881,2104-09-24 17:31:00,2104-09-30 16:17:00,1,1,2,2,0,7,1.0,5.948611,0.279861,value itemid 1 0.0 2 ...,amount itemid 1 0.0 2...,duration itemid 1 ...,M,1.9252,53


### Data Augmentation

In [72]:
# Data Augmentation 
df = master_df

df['ed_duration'] = df['ed_duration'].clip(lower=0)
df = df[df['age'] <= 100]
df = df[df['age'] >= 1]
df = df[df['los'] <= 30]
df = df[df['admit_duration'] <= 30]
df = df[df['ed_duration'] <= 1]
df['ed_duration'] = df['ed_duration'].clip(lower=0)

In [73]:
df.head()

,subject_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,marital_status,ethnicity,read_flag,admit_duration,ed_duration,event,input,procedure,gender,los,age
0,42231,2102-08-29 07:15:00,2102-09-06 16:20:00,0,2,2,1,0,7,0.0,8.378472,0.000000,value itemid 1 0.00...,amount itemid 1 0.0 2...,duration itemid 1 ...,F,1.1358,86
1,43881,2104-09-24 17:31:00,2104-09-30 16:17:00,1,1,2,2,0,7,1.0,5.948611,0.279861,value itemid 1 0.0 2 ...,amount itemid 1 0.0 2...,duration itemid 1 ...,M,1.9252,53
2,43879,2106-08-30 15:43:00,2106-08-31 15:15:00,1,0,1,1,0,2,0.0,0.980556,0.000000,value itemid 1 ...,amount itemid 1 0.0 2...,duration itemid 1 ...,F,0.9775,55
3,10088,2107-01-04 11:59:00,2107-01-11 15:45:00,1,1,8,1,1,7,1.0,7.156944,0.160417,value itemid 1 ...,amount itemid 1 0...,duration itemid 1 ...,M,3.6885,78
4,10061,2107-01-16 11:33:00,2107-02-10 11:30:00,1,3,7,1,1,7,0.0,24.997917,0.000000,value itemid 1 ...,amount itemid 1 ...,duration itemid 1 ...,F,24.9968,76
